In [ ]:
!pip install -q google-genai pydantic pandas

In [ ]:
from google.colab import userdata
from google import genai

# Read API key securely from Colab Secrets
API_KEY = userdata.get("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("GEMINI_API_KEY not found. Check Colab Secrets.")

client = genai.Client(api_key=API_KEY)

# Test Gemini Flash
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Reply with exactly: Gemini API connected successfully."
)

print(response.text)

In [ ]:
import os

required_files = [
    "appointments_clean.csv",
    "followup_notes.csv",
    "clinics.csv"
]

for file in required_files:
    if os.path.exists(file):
        print(f"FOUND: {file}")
    else:
        print(f"MISSING: {file}")

In [ ]:
import pandas as pd

appointments = pd.read_csv("appointments_clean.csv")
notes = pd.read_csv("followup_notes.csv")
clinics = pd.read_csv("clinics.csv")

print("Appointments:", appointments.shape)
print("Follow-up notes:", notes.shape)
print("Clinics:", clinics.shape)

print("\nFollow-up notes columns:")
print(notes.columns.tolist())

print("\nAppointments columns:")
print(appointments.columns.tolist())

print("\nClinics columns:")
print(clinics.columns.tolist())

In [ ]:
# Display the first 10 follow-up notes
first_10_notes = notes.head(10).copy()

pd.set_option("display.max_colwidth", None)

display(first_10_notes)

# Check missing or empty note text
print("\nEmpty note count:",
      first_10_notes["note_text"].fillna("").str.strip().eq("").sum())

# Check appointment ID matching
matched = first_10_notes["appointment_id"].isin(
    appointments["appointment_id"]
).sum()

print("Notes matched with appointments:", matched, "/ 10")

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

ReasonCategory = Literal[
    "No reminder",
    "Transport or distance",
    "Financial",
    "Illness",
    "Work or scheduling",
    "Contact issue",
    "Other",
    "Unknown"
]

class AnalysedNote(BaseModel):
    note_id: str
    appointment_id: str

    reason_category: ReasonCategory

    summary: str = Field(
        description="Short summary of the patient's explanation"
    )

    recommended_action: str = Field(
        description="Practical action the clinic should take"
    )

    confidence: Literal["High", "Medium", "Low"]

    is_empty: bool

class NotesAnalysis(BaseModel):
    analyses: list[AnalysedNote]

print("Pydantic schema created successfully!")
print(NotesAnalysis.model_json_schema())

In [ ]:
from pydantic import ValidationError

# Test 1: Missing required fields
invalid_response_1 = {
    "note_id": "N001",
    "appointment_id": "A0872",
    "reason_category": "No reminder"
}

# Test 2: Invalid reason category
invalid_response_2 = {
    "note_id": "N002",
    "appointment_id": "A0897",
    "reason_category": "Transportation problem",
    "summary": "Patient could not find transport.",
    "recommended_action": "Offer transport assistance.",
    "confidence": "High",
    "is_empty": False
}

# Validate both responses
for number, response in enumerate(
    [invalid_response_1, invalid_response_2], start=1
):
    try:
        AnalysedNote.model_validate(response)
        print(f"Test {number}: Unexpectedly passed")
    except ValidationError as error:
        print(f"Test {number}: INVALID RESPONSE CAUGHT")
        print(error.errors()[0]["msg"])

print("\nInvalid response testing completed.")

In [ ]:
import json

# Prepare the first 10 notes
notes_for_gemini = first_10_notes.copy()

notes_for_gemini["note_text"] = (
    notes_for_gemini["note_text"].fillna("")
)

notes_json = notes_for_gemini.to_dict(orient="records")

# CRAFT prompt
craft_prompt = f"""
CONTEXT:
CareLoop Clinics wants to understand why patients miss appointments.
Analyse the first 10 patient follow-up notes.

ROLE:
You are a healthcare operations data analyst.

ACTION:
For each note:
1. Identify the main reason for the missed appointment.
2. Write a short, factual summary.
3. Recommend a practical clinic action.
4. Assign confidence as High, Medium, or Low.
5. Set is_empty to true if the note contains no text.
6. For empty notes, use reason_category "Unknown".
7. Understand Hindi text as well as English.
8. Do not invent information.

FORMAT:
Return valid JSON only, following this structure:
{json.dumps(NotesAnalysis.model_json_schema(), indent=2)}

TONE:
Professional, neutral, concise and evidence-based.

PATIENT NOTES:
{json.dumps(notes_json, ensure_ascii=False, indent=2)}
"""

print("CRAFT prompt prepared successfully.")
print("Notes included:", len(notes_json))
print("Prompt characters:", len(craft_prompt))

In [ ]:
import time
import json
from google.genai import types

gemini_success = False

for attempt in range(1, 6):
    try:
        print(f"Gemini attempt {attempt}/5...")

        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents=craft_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            )
        )

        # Validate Gemini's JSON response
        validated_result = NotesAnalysis.model_validate_json(
            response.text
        )

        # Save raw response
        with open("gemini_raw_response.json", "w") as f:
            f.write(response.text)

        # Convert validated results into DataFrame
        analysed_data = [
            item.model_dump()
            for item in validated_result.analyses
        ]

        notes_analysed = pd.DataFrame(analysed_data)
        notes_analysed.to_csv(
            "notes_analysed.csv", index=False
        )

        print("Gemini analysis successful!")
        print("Notes analysed:", len(notes_analysed))
        print("Saved: notes_analysed.csv")
        gemini_success = True
        break

    except Exception as e:
        print("Attempt failed:", str(e)[:250])

        if attempt < 5:
            wait = 2 ** attempt
            print(f"Retrying in {wait} seconds...")
            time.sleep(wait)

if not gemini_success:
    print("\nGemini is still unavailable.")
    print("We can continue with the remaining offline tasks.")


In [ ]:
print("Analysed notes:")
display(notes_analysed)

print("\nReason category counts:")
print(notes_analysed["reason_category"].value_counts())

print("\nEmpty notes handled:")
print(notes_analysed[["note_id", "reason_category", "is_empty"]])

print("\nTotal records:", len(notes_analysed))
print("Unique note IDs:", notes_analysed["note_id"].nunique())
print("Missing summaries:", notes_analysed["summary"].isna().sum())

In [ ]:
# Merge analysed notes with appointments
merged = notes_analysed.merge(
    appointments,
    on="appointment_id",
    how="left",
    validate="one_to_one"
)

# Add clinic information
merged = merged.merge(
    clinics,
    on="clinic_id",
    how="left",
    validate="many_to_one"
)

# Check merge results
print("Merged dataset shape:", merged.shape)
print("Matched appointments:", merged["clinic_id"].notna().sum())
print("Matched clinics:", merged["clinic_name"].notna().sum())

# Save merged dataset
merged.to_csv("merged_notes_appointments_clinics.csv", index=False)

# Display important columns
display(
    merged[
        ["note_id", "appointment_id", "clinic_name",
         "city", "reason_category", "no_show"]
    ]
)

In [ ]:
# Cross-tabulation: reason by clinic
clinic_crosstab = pd.crosstab(
    merged["clinic_name"],
    merged["reason_category"]
)

# Cross-tabulation: reason by city
city_crosstab = pd.crosstab(
    merged["city"],
    merged["reason_category"]
)

print("REASONS BY CLINIC")
display(clinic_crosstab)

print("\nREASONS BY CITY")
display(city_crosstab)

# Save both tables
clinic_crosstab.to_csv("reason_by_clinic.csv")
city_crosstab.to_csv("reason_by_city.csv")

print("\nBoth cross-tabulation files saved successfully.")

In [ ]:
recommendations = {
    "Vaishali Nagar Clinic": (
        "Verify SMS reminder delivery and confirm patient contact preferences."
    ),
    "Vijay Nagar Clinic": (
        "Explore transport assistance and telehealth for routine visits."
    ),
    "Malviya Nagar Clinic": (
        "Offer evening and weekend appointments, review follow-up pricing, "
        "and contact patients whose reasons are unknown."
    ),
    "Gomti Nagar Care Point": (
        "Improve automated reminders and confirm successful SMS delivery."
    ),
    "Hadapsar Care Centre": (
        "Explore alternative transport options and telehealth appointments."
    ),
    "Kothrud Family Clinic": (
        "Collect alternate contact details and provide additional outreach channels."
    )
}

recommendation_df = pd.DataFrame(
    list(recommendations.items()),
    columns=["clinic_name", "recommendation"]
)

display(recommendation_df)

recommendation_df.to_csv(
    "clinic_specific_recommendations.csv",
    index=False
)

print("Clinic recommendations saved successfully.")

In [ ]:
basic_prompt = f"""
Analyse these 10 patient follow-up notes.
For each note, identify the reason for the missed appointment,
summarise it, and suggest an action for the clinic.

Return JSON containing an analyses list.
Each item should contain:
note_id, appointment_id, reason_category, summary,
recommended_action, confidence, is_empty.

Use the same reason categories as the Pydantic schema.
Do not invent information.

Notes:
{json.dumps(notes_json, ensure_ascii=False)}
"""

try:
    basic_response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=basic_prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        )
    )

    basic_result = NotesAnalysis.model_validate_json(
        basic_response.text
    )

    print("Basic prompt analysis successful!")
    print("Basic prompt records:", len(basic_result.analyses))
    print("CRAFT prompt records:", len(validated_result.analyses))

    print("\nPROMPT COMPARISON")
    print("CRAFT: Explicit context, role, action, format and tone.")
    print("Basic: Direct instruction with a short output specification.")
    print("Both outputs passed Pydantic validation.")

    with open("basic_prompt_response.json", "w") as f:
        f.write(basic_response.text)

except Exception as e:
    print("Basic prompt comparison error:", str(e)[:500])

In [ ]:
import time

basic_success = False

for attempt in range(1, 4):
    try:
        print(f"Basic prompt attempt {attempt}/3...")

        basic_response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents=basic_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            )
        )

        basic_result = NotesAnalysis.model_validate_json(
            basic_response.text
        )

        with open("basic_prompt_response.json", "w") as f:
            f.write(basic_response.text)

        print("\nBasic prompt analysis successful!")
        print("Records analysed:", len(basic_result.analyses))
        print("CRAFT records:", len(validated_result.analyses))
        print("Basic records:", len(basic_result.analyses))

        basic_success = True
        break

    except Exception as e:
        print("Error:", str(e)[:200])

        if attempt < 3:
            wait = attempt * 15
            print(f"Waiting {wait} seconds before retry...")
            time.sleep(wait)

if not basic_success:
    print("\nBasic prompt comparison still pending.")

In [ ]:
import json

try:
    raw_basic = json.loads(basic_response.text)

    print("Raw basic response recovered!")

    for item in raw_basic.get("analyses", []):
        print(
            item.get("note_id"),
            "=>",
            item.get("reason_category")
        )

except Exception as e:
    print("Could not recover response:", str(e))

In [ ]:
comparison = {
    "comparison": "CRAFT vs Basic Prompt",
    "notes_analysed": 10,
    "craft_prompt": {
        "valid_records": 10,
        "schema_validation": "Passed",
        "consistent_categories": True
    },
    "basic_prompt": {
        "records_returned": 10,
        "schema_validation": "Failed",
        "category_consistency": "Inconsistent",
        "issues": [
            "Used different category labels",
            "N004 had no reason category",
            "Pydantic rejected the response"
        ]
    },
    "conclusion": (
        "The CRAFT prompt produced more consistent structured output "
        "and passed Pydantic validation. The basic prompt identified "
        "similar reasons but failed the predefined schema."
    )
}

with open("prompt_comparison_report.json", "w") as f:
    json.dump(comparison, f, indent=4)

print("Prompt comparison report saved successfully!")

In [ ]:
import os
import json
import zipfile

from pathlib import Path

# Create submission folder
output_dir = Path("CareLoop_Part4_Submission")
output_dir.mkdir(exist_ok=True)

# Save the raw basic prompt response, even though validation failed
with open(output_dir / "basic_prompt_response.json", "w") as f:
    f.write(basic_response.text)

# Save the comparison report
with open(output_dir / "prompt_comparison_report.json", "w") as f:
    json.dump(comparison, f, indent=4)

# Create README
readme = """
# CareLoop Clinics — Part 4: Generative AI Analysis

## Overview
Gemini Flash was used to analyse the first 10 patient follow-up notes.

## Task 4.1 — Gemini API and CRAFT Prompt
- Integrated Gemini Flash using the google-genai SDK.
- Prepared a CRAFT prompt (Context, Role, Action, Format, Tone).
- Analysed 10 follow-up notes.
- Saved the raw Gemini response.

## Task 4.2 — Pydantic Validation
- Created Pydantic models for structured responses.
- Validated all 10 CRAFT results.
- Generated notes_analysed.csv.
- Handled the empty N004 note as Unknown.

## Task 4.3 — Validation and Prompt Comparison
- Tested two deliberately invalid responses.
- Both were rejected by Pydantic.
- Compared CRAFT and basic prompt outputs.
- CRAFT output passed schema validation.
- Basic output used inconsistent reason labels and failed
  strict schema validation.
- Both raw outputs and the comparison report are included.

## Task 4.4 — Clinic Analysis
- Merged notes with appointments and clinics.
- All 10 appointment and clinic records matched.
- Generated reason-by-clinic and reason-by-city tables.
- Generated clinic-specific recommendations.

## Reason Summary
- No reminder: 2
- Transport or distance: 2
- Work or scheduling: 2
- Financial: 1
- Illness: 1
- Contact issue: 1
- Unknown: 1

These are findings from the first 10 notes only.
They are not population-wide clinic rates.

## Files
- gemini_raw_response.json
- basic_prompt_response.json
- notes_analysed.csv
- merged_notes_appointments_clinics.csv
- reason_by_clinic.csv
- reason_by_city.csv
- clinic_specific_recommendations.csv
- prompt_comparison_report.json

## Security
Never publish API keys or credentials.
Use Colab Secrets for API credentials.
"""

(output_dir / "README.md").write_text(readme)

# Copy generated output files into submission folder
files_to_copy = [
    "gemini_raw_response.json",
    "notes_analysed.csv",
    "merged_notes_appointments_clinics.csv",
    "reason_by_clinic.csv",
    "reason_by_city.csv",
    "clinic_specific_recommendations.csv",
    "prompt_comparison_report.json"
]

for filename in files_to_copy:
    if os.path.exists(filename):
        with open(filename, "rb") as src:
            (output_dir / filename).write_bytes(src.read())
    else:
        print("WARNING — file not found:", filename)

# Create ZIP
zip_name = "CareLoop_Part4_Submission.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for file in output_dir.iterdir():
        if file.is_file():
            z.write(file, arcname=f"CareLoop_Part4_Submission/{file.name}")

print("\nPART 4 PACKAGE CREATED")
print("Files included:")
for file in sorted(output_dir.iterdir()):
    print("-", file.name)

print("\nZIP:", zip_name)
print("Size:", round(os.path.getsize(zip_name) / 1024, 2), "KB")

In [ ]:
# Check the saved Basic Prompt response
import json
from pathlib import Path

print("Basic Prompt response available:",
      "basic_response" in globals())

print("Comparison report available:",
      Path("prompt_comparison_report.json").exists())

print("Analysed notes available:",
      Path("notes_analysed.csv").exists())

In [ ]:
import json

# Display the original Basic Prompt response
print("ORIGINAL BASIC PROMPT RESPONSE:")
print(json.dumps(basic_response.text, indent=2, ensure_ascii=False))

print("\nEXISTING PROMPT COMPARISON:")
with open("prompt_comparison_report.json", "r") as f:
    comparison_report = json.load(f)

print(json.dumps(comparison_report, indent=2, ensure_ascii=False))

In [ ]:
import json
from pathlib import Path

# Parse original Basic Prompt response
basic_data = json.loads(basic_response.text)

# Map Basic Prompt categories to the predefined categories
category_map = {
    "FORGOT_NO_REMINDER": "No reminder",
    "TRANSPORTATION": "Transport or distance",
    "FINANCIAL": "Financial",
    "ILLNESS": "Illness",
    "WORK_COMMITMENT": "Work or scheduling",
    "UNREACHABLE": "Contact issue"
}

normalized_records = []

for record in basic_data["analyses"]:
    confidence_score = record.get("confidence", 0)

    if confidence_score >= 0.85:
        confidence = "High"
    elif confidence_score >= 0.60:
        confidence = "Medium"
    else:
        confidence = "Low"

    normalized_records.append({
        "note_id": record["note_id"],
        "appointment_id": record["appointment_id"],
        "reason_category": category_map.get(
            record.get("reason_category"), "Unknown"
        ),
        "summary": record.get("summary") or "No note text available.",
        "recommended_action": record.get("recommended_action") or
                              "Contact patient to clarify the reason.",
        "confidence": confidence,
        "is_empty": record.get("is_empty", False)
    })

# Save normalized response separately
with open("basic_prompt_normalized.json", "w") as f:
    json.dump(
        {"analyses": normalized_records},
        f,
        indent=2,
        ensure_ascii=False
    )

print("Normalized records:", len(normalized_records))
print("Categories:")

from collections import Counter
print(Counter(r["reason_category"] for r in normalized_records))

print("\nSaved: basic_prompt_normalized.json")

In [ ]:
import json
from pathlib import Path

# Validate every record using the existing Pydantic model
validated_basic = []

for record in normalized_records:
    if hasattr(AnalysedNote, "model_validate"):
        validated = AnalysedNote.model_validate(record)
    else:
        validated = AnalysedNote.parse_obj(record)

    validated_basic.append(
        validated.model_dump() if hasattr(validated, "model_dump")
        else validated.dict()
    )

# Save validation results
validation_report = {
    "prompt": "Basic Prompt",
    "total_records": len(normalized_records),
    "valid_records": len(validated_basic),
    "schema_validation": "Passed",
    "invalid_records": 0
}

with open("basic_prompt_validation_report.json", "w") as f:
    json.dump(validation_report, f, indent=2)

print("BASIC PROMPT VALIDATION")
print(json.dumps(validation_report, indent=2))

In [ ]:
import json

with open("prompt_comparison_report.json", "r") as f:
    report = json.load(f)

report["basic_prompt"]["normalized_records"] = 10
report["basic_prompt"]["normalized_schema_validation"] = "Passed"
report["basic_prompt"]["normalization_required"] = True
report["basic_prompt"]["normalization_details"] = [
    "Mapped inconsistent category labels to predefined categories",
    "Converted numeric confidence scores to High/Medium/Low",
    "Preserved the empty note as Unknown"
]

report["conclusion"] = (
    "The CRAFT prompt generated consistent structured output "
    "and passed Pydantic validation directly. The Basic Prompt "
    "identified similar reasons but initially failed validation "
    "because its category labels and confidence format differed "
    "from the predefined schema. After normalization, all 10 "
    "Basic Prompt records passed validation. This demonstrates "
    "that CRAFT reduced the need for post-processing."
)

with open("prompt_comparison_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Updated comparison report successfully!")
print(json.dumps(report, indent=2))

In [ ]:
import zipfile
from pathlib import Path

files_to_include = [
    "README.md",
    "basic_prompt_response.json",
    "basic_prompt_normalized.json",
    "basic_prompt_validation_report.json",
    "clinic_specific_recommendations.csv",
    "gemini_raw_response.json",
    "merged_notes_appointments_clinics.csv",
    "notes_analysed.csv",
    "prompt_comparison_report.json",
    "reason_by_city.csv",
    "reason_by_clinic.csv"
]

missing = [f for f in files_to_include if not Path(f).exists()]

if missing:
    print("Missing files:", missing)
else:
    zip_name = "CareLoop_Part4_Submission.zip"

    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
        for file in files_to_include:
            z.write(file)

    print("PART 4 ZIP CREATED SUCCESSFULLY")
    print("Files included:", len(files_to_include))
    print("ZIP size:", round(Path(zip_name).stat().st_size / 1024, 2), "KB")

    with zipfile.ZipFile(zip_name, "r") as z:
        print("\nZIP CONTENTS:")
        for file in z.namelist():
            print("-", file)

In [ ]:
import os

print("Current folder:", os.getcwd())
print("\nFiles currently available:")

for item in os.listdir("."):
    print(item)

print("\nBasic response variable available:",
      "basic_response" in globals())

print("Normalized records available:",
      "normalized_records" in globals())

In [ ]:
import json
from pathlib import Path

# 1. Restore original Basic Prompt response
with open("basic_prompt_response.json", "w") as f:
    json.dump(
        json.loads(basic_response.text),
        f,
        indent=2,
        ensure_ascii=False
    )

# 2. Recreate README
readme = """# CareLoop Clinics — Part 4: Generative AI Analysis

## Objective
Analyse patient follow-up notes using Gemini and structured prompting.

## Dataset
- appointments_clean.csv
- followup_notes.csv
- clinics.csv

## Tasks Completed
1. Loaded and inspected the datasets.
2. Defined a Pydantic schema for structured note analysis.
3. Tested invalid responses and schema validation.
4. Created a CRAFT prompt for Gemini.
5. Analysed 10 follow-up notes using Gemini.
6. Validated the generated structured output.
7. Merged analysed notes with appointment and clinic data.
8. Created reason-by-clinic and reason-by-city summaries.
9. Generated clinic-specific recommendations.
10. Compared CRAFT and Basic Prompt outputs.

## Prompt Comparison
The CRAFT prompt produced 10 valid records and passed schema validation directly.

The Basic Prompt initially used inconsistent category labels and numeric
confidence scores. It required normalization into the predefined categories
and confidence levels. After normalization, all 10 records passed validation.

This demonstrates that CRAFT reduced the need for post-processing.

## Important Outputs
- notes_analysed.csv
- gemini_raw_response.json
- basic_prompt_response.json
- basic_prompt_normalized.json
- basic_prompt_validation_report.json
- merged_notes_appointments_clinics.csv
- reason_by_clinic.csv
- reason_by_city.csv
- clinic_specific_recommendations.csv
- prompt_comparison_report.json

## API Security
The Gemini API key was stored using Colab Secrets and is not included
in the submission.
"""

Path("README.md").write_text(readme, encoding="utf-8")

print("Recovered README.md")
print("Recovered basic_prompt_response.json")
print("Both files exist:",
      Path("README.md").exists() and
      Path("basic_prompt_response.json").exists())

In [ ]:
import zipfile
from pathlib import Path

files_to_include = [
    "README.md",
    "basic_prompt_response.json",
    "basic_prompt_normalized.json",
    "basic_prompt_validation_report.json",
    "clinic_specific_recommendations.csv",
    "gemini_raw_response.json",
    "merged_notes_appointments_clinics.csv",
    "notes_analysed.csv",
    "prompt_comparison_report.json",
    "reason_by_city.csv",
    "reason_by_clinic.csv"
]

zip_name = "CareLoop_Part4_Submission.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in files_to_include:
        if Path(filename).exists():
            z.write(filename)
        else:
            print("MISSING:", filename)

with zipfile.ZipFile(zip_name, "r") as z:
    contents = z.namelist()

print("\nFINAL ZIP VERIFICATION")
print("Files included:", len(contents))
print("Expected files:", len(files_to_include))
print("All files present:", len(contents) == len(files_to_include))

for filename in contents:
    print("✓", filename)

In [ ]:
from google.colab import files
files.download("CareLoop_Part4_Submission.zip")